In [2]:
from datasets import load_from_disk
import numpy as np
import pickle

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load (wrong) Huggingface dataset
data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_old.dataset'
ds = load_from_disk(data_path)
ds

In [ ]:
# Load (fixed) Huggingface dataset
data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
ds = load_from_disk(data_path)
ds

In [ ]:
def _reorder_pair(xs, ys):
    # Stable sort by squared radius without NumPy overhead
    order = sorted(range(len(xs)), key=lambda i: xs[i]*xs[i] + ys[i]*ys[i])
    return [xs[i] for i in order], [ys[i] for i in order]

def reorder_batch(batch):
    rx, ry = batch["rel_x_coord"], batch["rel_y_coord"]
    new_x, new_y = [], []
    append_x, append_y = new_x.append, new_y.append  # tiny micro-opt
    for xs, ys in zip(rx, ry):
        xs_sorted, ys_sorted = _reorder_pair(xs, ys)
        append_x(xs_sorted); append_y(ys_sorted)
    batch["rel_x_coord"] = new_x
    batch["rel_y_coord"] = new_y
    return batch

# Tune these for your machine and IO:
NUM_PROC = 208          # e.g., number of CPU cores
READ_BATCH = 1000     # how many rows processed per function call
WRITE_BATCH = 10000    # how many rows written per Arrow chunk

ds = ds.map(
    reorder_batch,
    batched=True,
    batch_size=READ_BATCH,
    writer_batch_size=WRITE_BATCH,
    num_proc=NUM_PROC,
    desc="Reordering rel_x_coord / rel_y_coord by radius"
)

In [ ]:
item = ds[0]

In [ ]:
item['rel_x_coord']

In [ ]:
item['rel_y_coord']

In [ ]:
data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_new.dataset'
ds.save_to_disk(data_path, num_shards=4000)
print("Saved to:", data_path)

In [ ]:
# Retrieve indices for reordering
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl', "rb") as f: 
    indices = pickle.load(f)

ds_all_indices = list(range(len(ds)))

# Add test and val indices at end of training indices
seen = set(indices)
to_add = []
for idx in ds_all_indices:
    if idx not in seen:
        to_add.append(idx)
        seen.add(idx)

indices.extend(to_add)

In [ ]:
cache_name = "/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_loader_reidx.arrow"
ds_ordered = ds.select(indices,
                       indices_cache_file_name=cache_name)

In [ ]:
# Materialize the new physical order
ds_ordered = ds_ordered.flatten_indices(num_proc=128, writer_batch_size=10000)

In [ ]:
new_path = data_path.replace('.dataset', '_ordered.dataset')
ds_ordered.save_to_disk(new_path, num_shards=4000)
print("Saved to:", new_path)

In [ ]:
print("All Done!")

In [ ]:
# Retrieve indices for reordering
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl', "rb") as f: 
    indices = pickle.load(f)

In [ ]:
train_indices = [i for i in range(len(indices))]

In [ ]:
# Save the split as pickle file
with open(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered_precomputed_split_train.pkl', 'wb') as f:
    pickle.dump(train_indices, f)
print("Saved train split.")

In [ ]:
# Load dataset
dataset = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered.dataset')

In [ ]:
selected_data = dataset.select(train_indices)

In [ ]:
n_nonzero_tokens = list(selected_data['n_nonzero_tokens'])

output_path = f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered_n_nonzero_tokens_train.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(n_nonzero_tokens, f)

In [3]:
# Save the split as pickle file
with open(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered_precomputed_split_train.pkl', 'rb') as f:
    train_indices = pickle.load(f)

In [4]:
len(train_indices)

81361647

In [5]:
# Save the split as pickle file
with open(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered_n_nonzero_tokens_train.pkl', 'rb') as f:
    n_nonzero_tokens = pickle.load(f)

In [6]:
len(n_nonzero_tokens)

81361647

In [7]:
# Save the split as pickle file
with open(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_split_train.pkl', 'rb') as f:
    train_indices = pickle.load(f)

In [8]:
len(train_indices)

81361647

In [9]:
# Save the split as pickle file
with open(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_n_nonzero_tokens_train.pkl', 'rb') as f:
    n_nonzero_tokens = pickle.load(f)

In [10]:
len(n_nonzero_tokens)

81361647